# Agentic Intelligence for Integrated Energy and Thermal Management  
## A Decision-Centric Framework for Sustainable Systems

This notebook presents a clean, reproducible workflow for **integrated renewable-energy forecasting and thermal-aware operational decision support**. The core idea is to move beyond standalone prediction and build a unified pipeline that:  
1. learns renewable-energy output from weather and temporal signals,  
2. estimates thermodynamics-inspired thermal demand using lightweight surrogate modeling,  
3. ranks feature relevance with **entropy-, correlation-, and mutual-information-guided analysis**, and  
4. converts predictions into **utility-based actions** for storage, HVAC moderation, and load adaptation.

The technical pipeline combines **feature engineering**, **ensemble machine learning** (including CatBoost, HistGradientBoosting, LightGBM, ExtraTrees, and related baselines), **cross-validation**, **weighted ensembling**, **global explainability**, **ablation analysis**, and **thermal-aware decision metrics**. The results are important because they show how a forecasting workflow can be upgraded into a **decision-centric energy–thermal intelligence system** while remaining lightweight, interpretable, and deployable in a Colab/GitHub setting.

All processed data products, tables, figures, models, and a consolidated `output_summary.txt` file are saved automatically to **Google Drive** when available.


### Package installation  
This cell contains the optional package installation command for a fresh Colab session. Leave it commented if the environment already has the required libraries.


In [ ]:
# Optional in a fresh Colab session:
# !pip install xgboost lightgbm catboost lime dice-ml shap joblib -q


### Imports and reproducibility  
This cell imports the scientific Python stack, machine-learning models, and evaluation tools. It also fixes the random seed for reproducibility.


In [ ]:
import os
import json
import math
import random
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    median_absolute_error,
    explained_variance_score,
    max_error,
)
from sklearn.feature_selection import mutual_info_regression
from sklearn.inspection import permutation_importance

from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
)

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
random.seed(SEED)


### Drive mounting and output folders  
This cell safely mounts Google Drive in Colab when needed, creates the output folder structure, and defines paths for tables, figures, models, snapshots, and the summary text file.


In [ ]:
IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

MOUNT_POINT = Path("/content/drive")
MYDRIVE = MOUNT_POINT / "MyDrive"

if IN_COLAB:
    if os.path.ismount(str(MOUNT_POINT)):
        print("Google Drive is already mounted.")
    else:
        if MOUNT_POINT.exists() and any(MOUNT_POINT.iterdir()):
            print("Cleaning stale mountpoint before mounting Google Drive...")
            shutil.rmtree(MOUNT_POINT)
        print("Mounting Google Drive...")
        drive.mount(str(MOUNT_POINT))
else:
    print("Colab was not detected. The notebook will use local paths if Google Drive is unavailable.")

BASE_DIR = (MYDRIVE / "Outputs" / "AETD2_GitHub_Ready") if MYDRIVE.exists() else Path.cwd() / "AETD2_GitHub_Ready"
TABLE_DIR = BASE_DIR / "tables"
FIG_DIR = BASE_DIR / "figures"
MODEL_DIR = BASE_DIR / "models"

for directory in [BASE_DIR, TABLE_DIR, FIG_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

PROCESSED_DATA_PATH = BASE_DIR / "renewable_processed.csv"
SNAPSHOT_PATH = BASE_DIR / "publication_snapshot.json"
OUTPUT_SUMMARY_PATH = BASE_DIR / "output_summary.txt"

print(f"Base output directory: {BASE_DIR}")
print(f"Tables directory: {TABLE_DIR}")
print(f"Figures directory: {FIG_DIR}")
print(f"Models directory: {MODEL_DIR}")


### Dataset discovery and raw-data loading  
This cell searches common Google Drive and runtime locations for the renewable-energy dataset, loads the CSV, and prints a concise raw-data summary.


In [ ]:
candidate_filenames = [
    "Renewable.csv",
    "renewable.csv",
    "02 modern-renewable-energy-consumption.csv",
    "03 modern-renewable-prod.csv",
]

candidate_folders = []
if MYDRIVE.exists():
    candidate_folders.extend([
        MYDRIVE / "Datasets" / "Energy",
        MYDRIVE / "Datasets",
        MYDRIVE,
    ])
candidate_folders.extend([
    Path("/content"),
    Path("/mnt/data"),
    Path.cwd(),
])

DATASET_PATH = None

for folder in candidate_folders:
    if folder.exists():
        for fname in candidate_filenames:
            path = folder / fname
            if path.exists():
                DATASET_PATH = path
                break
    if DATASET_PATH is not None:
        break

if DATASET_PATH is None and MYDRIVE.exists():
    print("Dataset not found in common Google Drive folders. Searching recursively...")
    for fname in candidate_filenames:
        hits = list(MYDRIVE.rglob(fname))
        if hits:
            DATASET_PATH = hits[0]
            break

if DATASET_PATH is None:
    print("Dataset not found in standard locations. Searching /content and /mnt/data recursively...")
    for root in [Path("/content"), Path("/mnt/data"), Path.cwd()]:
        if root.exists():
            for fname in candidate_filenames:
                hits = list(root.rglob(fname))
                if hits:
                    DATASET_PATH = hits[0]
                    break
        if DATASET_PATH is not None:
            break

if DATASET_PATH is None:
    raise FileNotFoundError(
        "No renewable-energy CSV file was found. Place the dataset in Google Drive or the current runtime and rerun this cell."
    )

df_raw = pd.read_csv(DATASET_PATH)

print("[RAW DATA]")
print(f"Dataset path: {DATASET_PATH}")
print(f"Raw shape: {df_raw.shape}")
print(f"Raw columns: {list(df_raw.columns)}")
df_raw.head()


### Initial preprocessing  
This cell standardizes the time column when available, derives the minute feature, removes duplicate rows, and prepares the working dataframe.


In [ ]:
df = df_raw.copy()

if "Time" in df.columns:
    df["Time"] = pd.to_datetime(df["Time"], errors="coerce")
    df["minute"] = df["Time"].dt.minute.fillna(0).astype(int)
else:
    if "minute" not in df.columns:
        df["minute"] = 0

duplicates_removed = int(len(df) - len(df.drop_duplicates()))
df = df.drop_duplicates().reset_index(drop=True)

print("[PREPROCESSING]")
print(f"Removed duplicate rows: {duplicates_removed}")
print(f"Current shape: {df.shape}")


### Renewable-energy feature engineering  
This cell builds engineered predictors from solar, weather, and temporal variables. These features help the models capture nonlinear structure, cyclical behavior, and interpretable physical interactions.


In [ ]:
df["sunlight_ratio_safe"] = df["sunlightTime"] / (df["dayLength"] + 1e-6)
df["night_indicator"] = (df["isSun"] == 0).astype(int)
df["ghi_temp_interaction"] = df["GHI"] * df["temp"]
df["clear_sky_proxy"] = (1 - df["clouds_all"] / 100.0) * df["GHI"]
df["humidity_temp_interaction"] = df["humidity"] * df["temp"]
df["wind_pressure_interaction"] = df["wind_speed"] * df["pressure"]

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["minute_sin"] = np.sin(2 * np.pi * df["minute"] / 60)
df["minute_cos"] = np.cos(2 * np.pi * df["minute"] / 60)


### Thermal-load surrogate engineering  
This cell adds a lightweight thermodynamics-inspired thermal branch. It estimates heating stress, cooling stress, thermal inertia, and comfort deviation to support downstream thermal-aware decision logic without requiring large external thermal datasets.


In [ ]:
T_SET_HEAT = 21.0
T_SET_COOL = 24.0
OCCUPANCY_FACTOR_DAY = 1.0
OCCUPANCY_FACTOR_NIGHT = 0.6

df["occupancy_proxy"] = np.where(
    (df["hour"] >= 7) & (df["hour"] <= 22),
    OCCUPANCY_FACTOR_DAY,
    OCCUPANCY_FACTOR_NIGHT,
)

df["heating_degree_gap"] = np.maximum(0, T_SET_HEAT - df["temp"])
df["cooling_degree_gap"] = np.maximum(0, df["temp"] - T_SET_COOL)

df["heating_load_proxy"] = df["heating_degree_gap"] * df["occupancy_proxy"]
df["cooling_load_proxy"] = df["cooling_degree_gap"] * df["occupancy_proxy"]

thermal_memory = 0.6 * (
    df["heating_load_proxy"].shift(1).fillna(0) + df["cooling_load_proxy"].shift(1).fillna(0)
)
df["thermal_inertia_proxy"] = thermal_memory

df["comfort_gap"] = np.where(
    df["temp"] < T_SET_HEAT,
    T_SET_HEAT - df["temp"],
    np.where(df["temp"] > T_SET_COOL, df["temp"] - T_SET_COOL, 0),
)

df["thermal_load_wh"] = (
    120 * df["heating_load_proxy"]
    + 140 * df["cooling_load_proxy"]
    + 40 * df["thermal_inertia_proxy"]
)

print("[THERMAL FEATURES]")
print(df[[
    "heating_load_proxy",
    "cooling_load_proxy",
    "thermal_inertia_proxy",
    "comfort_gap",
    "thermal_load_wh"
]].describe())


### Persist the processed dataset  
This cell saves the engineered dataset to Google Drive so downstream runs and GitHub users can inspect the final structured data file directly.


In [ ]:
df.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"Processed dataset saved to: {PROCESSED_DATA_PATH}")
print(f"Processed shape: {df.shape}")


### Feature matrix and target definition  
This cell defines the regression target and the model feature matrix. It also prints a compact target summary for later interpretation.


In [ ]:
TARGET = "Energy delta[Wh]"

feature_cols = [c for c in df.columns if c not in [TARGET, "Time"]]
X = df[feature_cols].copy()
y = df[TARGET].copy()

print("[TARGET SUMMARY]")
print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(y.describe())


### Correlation analysis  
This cell computes feature–target correlations and saves the resulting table. It is useful for quick physical sanity checks and early feature screening.


In [ ]:
corr_series = (
    pd.concat([X, y], axis=1)
    .corr(numeric_only=True)[TARGET]
    .drop(TARGET)
)
corr_df = corr_series.reset_index()
corr_df.columns = ["Feature", "Correlation"]
corr_df["AbsCorrelation"] = corr_df["Correlation"].abs()
corr_df = corr_df.sort_values("AbsCorrelation", ascending=False).reset_index(drop=True)
corr_df.to_csv(TABLE_DIR / "feature_correlations.csv", index=False)

print("[CORRELATION ANALYSIS]")
print(corr_df.head(10))


### Mutual-information analysis  
This cell estimates nonlinear feature relevance using mutual information and saves the ranked table for later comparison against entropy-based scores.


In [ ]:
mi_scores = mutual_info_regression(X, y, random_state=SEED)
mi_df = pd.DataFrame({"Feature": X.columns, "MI": mi_scores}).sort_values("MI", ascending=False).reset_index(drop=True)
mi_df.to_csv(TABLE_DIR / "mutual_information_scores.csv", index=False)

print("[MUTUAL INFORMATION]")
print(mi_df.head(10))


### Entropy-guided feature relevance  
This cell computes Shannon entropy per feature and combines it with mutual information and absolute correlation into a composite relevance score. The result supports a theory-enhanced feature-ranking stage.


In [ ]:
def shannon_entropy_from_series(series, bins=20):
    values = pd.Series(series).replace([np.inf, -np.inf], np.nan).dropna()
    if values.nunique() <= 1:
        return 0.0
    hist, _ = np.histogram(values, bins=bins, density=False)
    hist = hist[hist > 0]
    probs = hist / hist.sum()
    return float(-np.sum(probs * np.log2(probs)))

entropy_rows = []
for col in X.columns:
    entropy_rows.append({
        "Feature": col,
        "Entropy": shannon_entropy_from_series(X[col]),
        "MI": float(mi_df.loc[mi_df["Feature"] == col, "MI"].values[0]),
        "AbsCorrelation": float(corr_df.loc[corr_df["Feature"] == col, "AbsCorrelation"].values[0]),
    })

entropy_df = pd.DataFrame(entropy_rows)
entropy_df["CompositeScore"] = (
    0.4 * (entropy_df["Entropy"] / (entropy_df["Entropy"].max() + 1e-9))
    + 0.4 * (entropy_df["MI"] / (entropy_df["MI"].max() + 1e-9))
    + 0.2 * (entropy_df["AbsCorrelation"] / (entropy_df["AbsCorrelation"].max() + 1e-9))
)
entropy_df = entropy_df.sort_values("CompositeScore", ascending=False).reset_index(drop=True)
entropy_df.to_csv(TABLE_DIR / "entropy_feature_scores.csv", index=False)

print("[ENTROPY-GUIDED FEATURE RANKING]")
print(entropy_df.head(15))


### Feature-set definitions for ablation  
This cell defines multiple feature subsets so the notebook can measure the contribution of entropy-guided selection and thermal modeling separately.


In [ ]:
top_entropy_features = entropy_df.head(20)["Feature"].tolist()

feature_sets = {
    "Baseline": [
        c for c in feature_cols
        if c not in [
            "thermal_load_wh",
            "heating_load_proxy",
            "cooling_load_proxy",
            "thermal_inertia_proxy",
            "comfort_gap",
        ]
    ],
    "Baseline+EntropyTop20": top_entropy_features,
    "Energy+Thermal": feature_cols,
}

for name, cols in feature_sets.items():
    print(f"{name}: {len(cols)} features")


### Train–test split  
This cell performs a chronological split to preserve temporal structure and avoid leakage from future observations into the training subset.


In [ ]:
split_index = int(len(X) * 0.60)

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()
y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)
train_df.to_csv(TABLE_DIR / "train_df.csv", index=False)
test_df.to_csv(TABLE_DIR / "test_df.csv", index=False)

print("[TRAIN-TEST SPLIT]")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")


### Model factory  
This cell defines all benchmark models in one place so the same configuration can be reused consistently for benchmarking, cross-validation, and ablation.


In [ ]:
def build_models(seed=42):
    return {
        "LinearRegression": LinearRegression(),
        "Ridge": Ridge(),
        "ElasticNet": ElasticNet(),
        "RandomForest": RandomForestRegressor(random_state=seed, n_jobs=-1),
        "ExtraTrees": ExtraTreesRegressor(random_state=seed, n_jobs=-1),
        "GradientBoosting": GradientBoostingRegressor(random_state=seed),
        "HistGradientBoosting": HistGradientBoostingRegressor(random_state=seed),
        "XGBoost": XGBRegressor(
            random_state=seed,
            objective="reg:squarederror",
            eval_metric="rmse",
            n_jobs=-1,
        ),
        "LightGBM": LGBMRegressor(random_state=seed, n_jobs=-1),
        "CatBoost": CatBoostRegressor(random_state=seed, verbose=0),
    }


### Benchmarking and ensemble forecasting  
This cell trains all benchmark models on the full engineered feature set, evaluates them on the test split, and saves a ranked performance table.


In [ ]:
models_dict = build_models(SEED)
benchmark_rows = []

for model_name, model in models_dict.items():
    print(f"Training {model_name} ...")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)

    benchmark_rows.append({
        "Model": model_name,
        "R2": r2_score(y_test, preds),
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mean_absolute_error(y_test, preds),
    })

benchmark_df = pd.DataFrame(benchmark_rows).sort_values("R2", ascending=False).reset_index(drop=True)
benchmark_df.to_csv(TABLE_DIR / "model_benchmark.csv", index=False)

print("[MODEL BENCHMARK]")
display(benchmark_df)


### Cross-validation robustness check  
This cell performs 5-fold cross-validation on the strongest benchmark models and saves a compact robustness table.


In [ ]:
top_models = benchmark_df.head(4)["Model"].tolist()
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv_rows = []

for name in top_models:
    model = build_models(SEED)[name]
    print(f"Running CV for {name} ...")

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "r2": "r2",
            "mae": "neg_mean_absolute_error",
            "mse": "neg_mean_squared_error",
        },
        n_jobs=-1,
    )

    cv_rows.append({
        "Model": name,
        "CV_R2_Mean": scores["test_r2"].mean(),
        "CV_R2_Std": scores["test_r2"].std(),
        "CV_MAE_Mean": -scores["test_mae"].mean(),
        "CV_RMSE_Mean": np.sqrt(-scores["test_mse"].mean()),
    })

cv_df = pd.DataFrame(cv_rows).sort_values("CV_R2_Mean", ascending=False).reset_index(drop=True)
cv_df.to_csv(TABLE_DIR / "cross_validation_summary.csv", index=False)

print("[CROSS-VALIDATION SUMMARY]")
display(cv_df)


### Weighted top-3 ensemble  
This cell builds a weighted ensemble from the three strongest benchmark models, computes the final test metrics, and stores the ensemble definition for reuse.


In [ ]:
top3 = benchmark_df.head(3)["Model"].tolist()
top3_models = {name: build_models(SEED)[name] for name in top3}

for name, model in top3_models.items():
    print(f"Fitting ensemble member: {name}")
    model.fit(X_train, y_train)

weights_raw = benchmark_df.head(3)["R2"].values
weights = weights_raw / weights_raw.sum()

preds_top3 = np.column_stack([top3_models[name].predict(X_test) for name in top3])
ensemble_preds = np.average(preds_top3, axis=1, weights=weights)

ensemble_mse = mean_squared_error(y_test, ensemble_preds)
ensemble_rmse = np.sqrt(ensemble_mse)

ensemble_metrics = pd.DataFrame([{
    "Model": "WeightedEnsembleTop3",
    "R2": r2_score(y_test, ensemble_preds),
    "MSE": ensemble_mse,
    "RMSE": ensemble_rmse,
    "MAE": mean_absolute_error(y_test, ensemble_preds),
    "MedianAE": median_absolute_error(y_test, ensemble_preds),
    "MaxError": max_error(y_test, ensemble_preds),
    "EVS": explained_variance_score(y_test, ensemble_preds),
}])

ensemble_metrics.to_csv(TABLE_DIR / "ensemble_metrics.csv", index=False)

joblib.dump(build_models(SEED)[benchmark_df.iloc[0]["Model"]].fit(X_train, y_train), MODEL_DIR / "best_single_model.joblib")
with open(MODEL_DIR / "ensemble_info.json", "w") as f:
    json.dump({"top3_models": top3, "weights": weights.tolist()}, f, indent=2)

print("[ENSEMBLE METRICS]")
print("Top-3 members:", top3)
print("Top-3 weights:", weights.tolist())
display(ensemble_metrics)


### Prediction diagnostics and error categories  
This cell builds the main test-results table, computes error categories, and saves a diagnostic CSV for later visualization and inspection.


In [ ]:
results_df = X_test.copy()
results_df["Actual_Energy_Wh"] = y_test.values
results_df["Predicted_Energy_Wh"] = ensemble_preds
results_df["Absolute_Error"] = np.abs(results_df["Actual_Energy_Wh"] - results_df["Predicted_Energy_Wh"])

def categorize_error(row):
    actual = row["Actual_Energy_Wh"]
    abs_err = row["Absolute_Error"]
    if actual == 0:
        return "Zero-Actual / Undefined % Error"
    pct = 100 * abs_err / actual
    if pct < 20:
        return "Very Low Error (<20%)"
    elif pct < 50:
        return "Low Error (20-50%)"
    elif pct < 60:
        return "Moderate Error (50-60%)"
    return "High Error (>60%)"

results_df["Error_Category"] = results_df.apply(categorize_error, axis=1)
results_df.to_csv(TABLE_DIR / "test_predictions_diagnostics.csv", index=False)

error_dist = (
    results_df["Error_Category"]
    .value_counts(normalize=True)
    .mul(100)
    .reset_index()
)
error_dist.columns = ["Error_Category", "Percent"]
error_dist.to_csv(TABLE_DIR / "error_category_distribution.csv", index=False)

print("[ERROR CATEGORY DISTRIBUTION]")
display(error_dist)


### Global feature importance  
This cell estimates permutation-based feature importance using the best single model and saves the ranked importance table.


In [ ]:
best_single_name = benchmark_df.iloc[0]["Model"]
best_single_model = build_models(SEED)[best_single_name]
best_single_model.fit(X_train, y_train)

perm = permutation_importance(
    best_single_model,
    X_test,
    y_test,
    n_repeats=5,
    random_state=SEED,
    n_jobs=-1,
)

global_imp_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": perm.importances_mean,
}).sort_values("Importance", ascending=False).reset_index(drop=True)

global_imp_df.to_csv(TABLE_DIR / "global_feature_importance.csv", index=False)

print("[GLOBAL FEATURE IMPORTANCE]")
display(global_imp_df.head(15))


### AET-D² thermal-aware evaluation  
This cell computes the thermal-aware evaluation subset, derives operational metrics such as power and TA-ESI, and prepares the data required for decision support.


In [ ]:
aet_df = results_df.head(1000).copy()

aet_df["Thermal_Load_Wh"] = df.loc[aet_df.index, "thermal_load_wh"].values
aet_df["Comfort_Gap"] = df.loc[aet_df.index, "comfort_gap"].values
aet_df["Power_kW"] = aet_df["Predicted_Energy_Wh"] / 250.0
aet_df["Distance_km_per_hr"] = aet_df["Predicted_Energy_Wh"] / 1.25

eps = 1e-6
aet_df["TA_ESI"] = aet_df["Predicted_Energy_Wh"] / (
    aet_df["Predicted_Energy_Wh"] + aet_df["Thermal_Load_Wh"] + eps
)

print("[AET-D2 PRE-DECISION METRICS]")
display(aet_df[[
    "Predicted_Energy_Wh",
    "Thermal_Load_Wh",
    "Power_kW",
    "TA_ESI",
    "Distance_km_per_hr",
]].head())


### Utility-based decision engine  
This cell scores candidate operational actions with a compact utility function and assigns the highest-utility action together with an associated IoT-oriented response.


In [ ]:
def utility_scores(pred_energy, thermal_load, ta_esi, comfort_gap):
    scores = {}

    scores["Energy Storage"] = (
        1.5 * ta_esi
        + 0.0004 * max(pred_energy - thermal_load, 0)
        - 0.2 * comfort_gap
    )

    scores["Normal Operation"] = (
        1.2 * ta_esi
        - 0.0002 * abs(pred_energy - thermal_load)
        - 0.1 * comfort_gap
    )

    scores["HVAC Moderation"] = (
        0.8 * (1 - ta_esi)
        + 0.4 * comfort_gap
        - 0.0001 * pred_energy
    )

    scores["Load Reduction"] = (
        0.9 * (1 - ta_esi)
        + 0.0002 * thermal_load
        + 0.2 * comfort_gap
    )

    return scores

selected_actions = []
iot_actions = []

for idx, row in aet_df.iterrows():
    scores = utility_scores(
        row["Predicted_Energy_Wh"],
        row["Thermal_Load_Wh"],
        row["TA_ESI"],
        row["Comfort_Gap"],
    )
    action = max(scores, key=scores.get)
    selected_actions.append(action)

    if action == "Energy Storage":
        iot_actions.append("Store excess energy; Prepare controlled charging window")
    elif action == "Normal Operation":
        iot_actions.append("Maintain current operating state")
    elif action == "HVAC Moderation":
        iot_actions.append("Adjust HVAC setpoint; Reduce cooling or heating intensity")
    else:
        iot_actions.append("Turn off non-critical devices; Prioritize essential loads")

aet_df["Selected_Action"] = selected_actions
aet_df["IoT_Actions"] = iot_actions
aet_df.to_csv(TABLE_DIR / "aetd2_results_first_1000.csv", index=False)

print("[AET-D2 DECISION RESULTS]")
display(aet_df[[
    "Predicted_Energy_Wh",
    "Thermal_Load_Wh",
    "Power_kW",
    "TA_ESI",
    "Selected_Action",
    "IoT_Actions",
]].head())


### Quantitative energy–thermal summary  
This cell aggregates the AET-D² decision results into a compact summary table that highlights average system conditions and action tendencies.


In [ ]:
quant_eval = pd.DataFrame([{
    "Records_Evaluated": len(aet_df),
    "Mean_Predicted_Energy_Wh": aet_df["Predicted_Energy_Wh"].mean(),
    "Mean_Thermal_Load_Wh": aet_df["Thermal_Load_Wh"].mean(),
    "Mean_Power_kW": aet_df["Power_kW"].mean(),
    "Mean_TA_ESI": aet_df["TA_ESI"].mean(),
    "Mean_Distance_km_per_hr": aet_df["Distance_km_per_hr"].mean(),
    "Stable_Ratio_percent": (aet_df["Selected_Action"] == "Normal Operation").mean() * 100,
    "Deficit_Ratio_percent": (aet_df["Predicted_Energy_Wh"] < aet_df["Thermal_Load_Wh"]).mean() * 100,
    "Storage_Action_Ratio_percent": (aet_df["Selected_Action"] == "Energy Storage").mean() * 100,
}])

quant_eval.to_csv(TABLE_DIR / "aetd2_quantitative_evaluation.csv", index=False)

print("[AET-D2 QUANTITATIVE EVALUATION]")
display(quant_eval)


### Ablation study  
This cell compares the baseline feature set, the entropy-reduced set, and the full energy-plus-thermal set using a consistent XGBoost configuration.


In [ ]:
ablation_rows = []

for config_name, cols in feature_sets.items():
    print(f"Running ablation config: {config_name} ({len(cols)} features)")
    model = XGBRegressor(
        random_state=SEED,
        objective="reg:squarederror",
        eval_metric="rmse",
        n_jobs=-1,
    )
    model.fit(X_train[cols], y_train)
    preds = model.predict(X_test[cols])

    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)

    ablation_rows.append({
        "Configuration": config_name,
        "Num_Features": len(cols),
        "R2": r2_score(y_test, preds),
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mean_absolute_error(y_test, preds),
    })

ablation_df = pd.DataFrame(ablation_rows).sort_values("R2", ascending=False).reset_index(drop=True)
ablation_df.to_csv(TABLE_DIR / "ablation_study.csv", index=False)

print("[ABLATION STUDY]")
display(ablation_df)


### Figure generation and export  
This cell creates a compact set of GitHub-friendly figures and saves them directly to Google Drive. The figures summarize benchmarking, cross-validation, global importance, prediction behavior, decision distribution, and ablation results.


In [ ]:
# Figure 1: Benchmark R2
plt.figure(figsize=(10, 5))
plt.bar(benchmark_df["Model"], benchmark_df["R2"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("R2")
plt.title("Model Benchmarking by R2")
plt.tight_layout()
benchmark_fig = FIG_DIR / "benchmark_r2.png"
plt.savefig(benchmark_fig, dpi=300, bbox_inches="tight")
plt.show()

# Figure 2: Cross-validation mean R2
plt.figure(figsize=(8, 4.5))
plt.bar(cv_df["Model"], cv_df["CV_R2_Mean"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("Mean CV R2")
plt.title("Cross-Validation Performance")
plt.tight_layout()
cv_fig = FIG_DIR / "cross_validation_r2.png"
plt.savefig(cv_fig, dpi=300, bbox_inches="tight")
plt.show()

# Figure 3: Actual vs predicted scatter
sample_plot_df = results_df.sample(min(5000, len(results_df)), random_state=SEED)
plt.figure(figsize=(6, 6))
plt.scatter(sample_plot_df["Actual_Energy_Wh"], sample_plot_df["Predicted_Energy_Wh"], alpha=0.25, s=10)
min_val = min(sample_plot_df["Actual_Energy_Wh"].min(), sample_plot_df["Predicted_Energy_Wh"].min())
max_val = max(sample_plot_df["Actual_Energy_Wh"].max(), sample_plot_df["Predicted_Energy_Wh"].max())
plt.plot([min_val, max_val], [min_val, max_val])
plt.xlabel("Actual Energy (Wh)")
plt.ylabel("Predicted Energy (Wh)")
plt.title("Actual vs Predicted Energy")
plt.tight_layout()
scatter_fig = FIG_DIR / "actual_vs_predicted_scatter.png"
plt.savefig(scatter_fig, dpi=300, bbox_inches="tight")
plt.show()

# Figure 4: Global feature importance
top_imp = global_imp_df.head(10).iloc[::-1]
plt.figure(figsize=(8, 5))
plt.barh(top_imp["Feature"], top_imp["Importance"])
plt.xlabel("Permutation Importance")
plt.title("Top Global Features")
plt.tight_layout()
importance_fig = FIG_DIR / "global_feature_importance.png"
plt.savefig(importance_fig, dpi=300, bbox_inches="tight")
plt.show()

# Figure 5: Error categories
plt.figure(figsize=(9, 4.5))
plt.bar(error_dist["Error_Category"], error_dist["Percent"])
plt.xticks(rotation=25, ha="right")
plt.ylabel("Percent")
plt.title("Error Category Distribution")
plt.tight_layout()
error_fig = FIG_DIR / "error_category_distribution.png"
plt.savefig(error_fig, dpi=300, bbox_inches="tight")
plt.show()

# Figure 6: Action distribution
action_counts = (
    aet_df["Selected_Action"]
    .value_counts()
    .reset_index()
)
action_counts.columns = ["Selected_Action", "Count"]
action_counts.to_csv(TABLE_DIR / "action_distribution.csv", index=False)

plt.figure(figsize=(8, 4.5))
plt.bar(action_counts["Selected_Action"], action_counts["Count"])
plt.xticks(rotation=20, ha="right")
plt.ylabel("Count")
plt.title("AET-D2 Action Distribution")
plt.tight_layout()
action_fig = FIG_DIR / "aetd2_action_distribution.png"
plt.savefig(action_fig, dpi=300, bbox_inches="tight")
plt.show()

# Figure 7: Ablation study R2
plt.figure(figsize=(8, 4.5))
plt.bar(ablation_df["Configuration"], ablation_df["R2"])
plt.xticks(rotation=20, ha="right")
plt.ylabel("R2")
plt.title("Ablation Study by Configuration")
plt.tight_layout()
ablation_fig = FIG_DIR / "ablation_r2.png"
plt.savefig(ablation_fig, dpi=300, bbox_inches="tight")
plt.show()

generated_figures = [
    benchmark_fig,
    cv_fig,
    scatter_fig,
    importance_fig,
    error_fig,
    action_fig,
    ablation_fig,
]

print("Saved figures:")
for fig_path in generated_figures:
    print(fig_path)


### Snapshot and summary export  
This cell writes a machine-readable JSON snapshot and a human-readable `output_summary.txt` file that lists the main results, tables, figures, and output locations.


In [ ]:
snapshot = {
    "dataset_path": str(DATASET_PATH),
    "processed_data_path": str(PROCESSED_DATA_PATH),
    "best_single_model": benchmark_df.iloc[0]["Model"],
    "best_single_r2": float(benchmark_df.iloc[0]["R2"]),
    "final_model": "WeightedEnsembleTop3",
    "ensemble_r2": float(ensemble_metrics.iloc[0]["R2"]),
    "top_features": global_imp_df.head(10).to_dict(orient="records"),
    "quantitative_evaluation": quant_eval.to_dict(orient="records")[0],
}
with open(SNAPSHOT_PATH, "w") as f:
    json.dump(snapshot, f, indent=2)

generated_tables = [
    TABLE_DIR / "train_df.csv",
    TABLE_DIR / "test_df.csv",
    TABLE_DIR / "feature_correlations.csv",
    TABLE_DIR / "mutual_information_scores.csv",
    TABLE_DIR / "entropy_feature_scores.csv",
    TABLE_DIR / "model_benchmark.csv",
    TABLE_DIR / "cross_validation_summary.csv",
    TABLE_DIR / "ensemble_metrics.csv",
    TABLE_DIR / "test_predictions_diagnostics.csv",
    TABLE_DIR / "error_category_distribution.csv",
    TABLE_DIR / "global_feature_importance.csv",
    TABLE_DIR / "aetd2_results_first_1000.csv",
    TABLE_DIR / "aetd2_quantitative_evaluation.csv",
    TABLE_DIR / "ablation_study.csv",
    TABLE_DIR / "action_distribution.csv",
]

lines = []
lines.append("AET-D2 GitHub-ready notebook - output summary")
lines.append("=" * 88)
lines.append(f"Dataset path: {DATASET_PATH}")
lines.append(f"Output directory: {BASE_DIR}")
lines.append(f"Processed dataset path: {PROCESSED_DATA_PATH}")
lines.append("")
lines.append("[MODEL BENCHMARK]")
lines.append(benchmark_df.to_string(index=False))
lines.append("")
lines.append("[CROSS-VALIDATION SUMMARY]")
lines.append(cv_df.to_string(index=False))
lines.append("")
lines.append("[ENSEMBLE METRICS]")
lines.append(ensemble_metrics.to_string(index=False))
lines.append("")
lines.append("[AET-D2 QUANTITATIVE EVALUATION]")
lines.append(quant_eval.to_string(index=False))
lines.append("")
lines.append("[ABLATION STUDY]")
lines.append(ablation_df.to_string(index=False))
lines.append("")
lines.append("[GENERATED TABLES]")
for path in generated_tables:
    lines.append(str(path))
lines.append("")
lines.append("[GENERATED FIGURES]")
for path in generated_figures:
    lines.append(str(path))
lines.append("")
lines.append("[GENERATED MODELS]")
lines.append(str(MODEL_DIR / "best_single_model.joblib"))
lines.append(str(MODEL_DIR / "ensemble_info.json"))
lines.append("")
lines.append("[SNAPSHOT]")
lines.append(str(SNAPSHOT_PATH))

with open(OUTPUT_SUMMARY_PATH, "w") as f:
    f.write("\n".join(lines))

print(f"Snapshot saved to: {SNAPSHOT_PATH}")
print(f"Summary saved to: {OUTPUT_SUMMARY_PATH}")


### Final note  
The notebook is now organized for public GitHub use: it contains a self-explanatory introduction, a subtitle and explanation before every code block, direct Google Drive export of tables and figures, and a consolidated `output_summary.txt` file for quick inspection of all outputs.
